# Raw vs Translation-Canonical Discovery on Heat Data

This notebook compares two discovery paths on the same synthetic Heat fields:

- raw path: `to_pysindy_trajectories(...) -> fit_pysindy_discovery(...)`
- translation-canonical path: `build_translation_canonical_discovery_inputs(...) -> fit_pysindy_discovery(...)`

The goal is not to claim exact PDE recovery. The `v0.6` adapter returns backend-native PySINDy terms. The interesting question is whether heuristic translation canonicalization makes the fit structurally cleaner or more repeatable.


In [ ]:
import importlib.util

if importlib.util.find_spec("pysindy") is None:
    raise RuntimeError("Install pdelie[downstream] or pdelie[test] to run this notebook.")

import numpy as np

from pdelie import GeneratorFamily
from pdelie.data import generate_heat_1d_field_batch
from pdelie.discovery import (
    build_translation_canonical_discovery_inputs,
    fit_pysindy_discovery,
    to_pysindy_trajectories,
)


In [ ]:
def make_known_translation_generator() -> GeneratorFamily:
    basis_spec = {
        "variables": ["t", "x", "u"],
        "component_names": ["xi"],
        "basis_terms": [
            {"label": "1", "powers": [0, 0, 0]},
            {"label": "t", "powers": [1, 0, 0]},
            {"label": "x", "powers": [0, 1, 0]},
            {"label": "u", "powers": [0, 0, 1]},
        ],
        "component_ordering": ["xi"],
        "term_ordering": ["1", "t", "x", "u"],
        "layout": "component_major",
    }
    return GeneratorFamily(
        parameterization="polynomial_translation_affine",
        coefficients=np.array([[1.0, 0.0, 0.0, 0.0]], dtype=float),
        basis_spec=basis_spec,
        normalization="l2_unit",
        diagnostics={},
    )


def count_nonzero_backend_terms(result: dict[str, object], feature_name: str) -> int:
    return len(result["equation_terms"].get(feature_name, {}))


def feature_score(result: dict[str, object], feature_name: str) -> tuple[int, float]:
    coefficients = result["coefficients"]
    if coefficients is None:
        return (0, 0.0)
    row_index = result["feature_names"].index(feature_name)
    row = coefficients[row_index]
    return (count_nonzero_backend_terms(result, feature_name), float(np.linalg.norm(row)))


def choose_feature_from_fit(raw_fit: dict[str, object], canonical_fit: dict[str, object], feature_names: list[str]) -> str:
    scored = []
    for feature_name in feature_names:
        raw_nnz, raw_norm = feature_score(raw_fit, feature_name)
        canonical_nnz, canonical_norm = feature_score(canonical_fit, feature_name)
        scored.append((raw_nnz + canonical_nnz, raw_norm + canonical_norm, feature_name))
    scored.sort(key=lambda item: (item[0], item[1], item[2]), reverse=True)
    if scored and (scored[0][0] > 0 or scored[0][1] > 0.0):
        return scored[0][2]
    return feature_names[len(feature_names) // 2]


def initial_batch_std_by_feature(trajectories) -> np.ndarray:
    stacked = np.stack(trajectories, axis=0)
    return np.std(stacked[:, 0, :], axis=0)


def choose_feature_to_inspect(raw: dict[str, object], canonical: dict[str, object]) -> str:
    raw_std = initial_batch_std_by_feature(raw["trajectories"])
    canonical_std = initial_batch_std_by_feature(canonical["inputs"]["trajectories"])
    reduction = raw_std - canonical_std
    best_index = int(np.argmax(reduction))
    if float(reduction[best_index]) > 0.0:
        return raw["feature_names"][best_index]
    return choose_feature_from_fit(raw["fit"], canonical["fit"], raw["feature_names"])


def initial_batch_std_summary(raw: dict[str, object], canonical: dict[str, object], feature_name: str) -> tuple[float, float, float]:
    feature_index = raw["feature_names"].index(feature_name)
    raw_std = initial_batch_std_by_feature(raw["trajectories"])
    canonical_std = initial_batch_std_by_feature(canonical["inputs"]["trajectories"])
    return (
        float(raw_std[feature_index]),
        float(canonical_std[feature_index]),
        float(raw_std[feature_index] - canonical_std[feature_index]),
    )


def top_dense_terms(result: dict[str, object], feature_name: str, *, top_k: int = 5):
    coefficients = result["coefficients"]
    if coefficients is None:
        return []
    row_index = result["feature_names"].index(feature_name)
    row = coefficients[row_index]
    order = np.argsort(np.abs(row))[::-1][:top_k]
    library_terms = result["library_feature_names"]
    return [
        (library_terms[index], float(row[index]))
        for index in order
        if abs(float(row[index])) > 0.0
    ]


def run_raw_fit(field):
    trajectories, time_values, feature_names = to_pysindy_trajectories(field)
    result = fit_pysindy_discovery(trajectories, time_values, feature_names)
    result["feature_names"] = feature_names
    return {
        "feature_names": feature_names,
        "trajectories": trajectories,
        "time_values": time_values,
        "fit": result,
    }


def run_canonical_fit(field, generator):
    inputs = build_translation_canonical_discovery_inputs(field, generator_family=generator)
    result = fit_pysindy_discovery(inputs["trajectories"], inputs["time_values"], inputs["feature_names"])
    result["feature_names"] = inputs["feature_names"]
    return {
        "inputs": inputs,
        "fit": result,
    }


In [ ]:
generator = make_known_translation_generator()
field = generate_heat_1d_field_batch(batch_size=4, num_times=33, num_points=64, seed=100)

raw = run_raw_fit(field)
canonical = run_canonical_fit(field, generator)

primary_feature = choose_feature_to_inspect(raw, canonical)
raw_primary_std, canonical_primary_std, primary_std_reduction = initial_batch_std_summary(raw, canonical, primary_feature)
summary = {
    "raw_status": raw["fit"]["status"],
    "canonical_status": canonical["fit"]["status"],
    "raw_coeff_shape": None if raw["fit"]["coefficients"] is None else tuple(raw["fit"]["coefficients"].shape),
    "canonical_coeff_shape": None if canonical["fit"]["coefficients"] is None else tuple(canonical["fit"]["coefficients"].shape),
    "inspected_feature": primary_feature,
    "raw_nonzero_primary_terms": count_nonzero_backend_terms(raw["fit"], primary_feature),
    "canonical_nonzero_primary_terms": count_nonzero_backend_terms(canonical["fit"], primary_feature),
    "raw_primary_row_l2": feature_score(raw["fit"], primary_feature)[1],
    "canonical_primary_row_l2": feature_score(canonical["fit"], primary_feature)[1],
    "raw_initial_batch_std": raw_primary_std,
    "canonical_initial_batch_std": canonical_primary_std,
    "initial_batch_std_reduction": primary_std_reduction,
    "canonical_alignment_policy": canonical["inputs"]["alignment_policy"],
    "canonical_alignment_shifts": canonical["inputs"]["alignment_shifts"][:4],
}
summary


## Repeatability across seeds

We compare the automatically selected inspected feature across several Heat seeds.
The notebook chooses the feature with the largest reduction in initial-time batch variability after canonicalization.
If the frozen adapter yields empty sparse equations under the default threshold, the batch-variability reduction and dense row norm still give useful structural signals.


In [ ]:
rows = []
for seed in [100, 101, 102, 103, 104]:
    field = generate_heat_1d_field_batch(batch_size=4, num_times=33, num_points=64, seed=seed)
    raw = run_raw_fit(field)
    canonical = run_canonical_fit(field, generator)
    feature_name = choose_feature_to_inspect(raw, canonical)
    raw_std, canonical_std, std_reduction = initial_batch_std_summary(raw, canonical, feature_name)
    rows.append(
        {
            "seed": seed,
            "raw_status": raw["fit"]["status"],
            "canonical_status": canonical["fit"]["status"],
            "inspected_feature": feature_name,
            "raw_nonzero_primary_terms": count_nonzero_backend_terms(raw["fit"], feature_name),
            "canonical_nonzero_primary_terms": count_nonzero_backend_terms(canonical["fit"], feature_name),
            "raw_primary_row_l2": feature_score(raw["fit"], feature_name)[1],
            "canonical_primary_row_l2": feature_score(canonical["fit"], feature_name)[1],
            "raw_initial_batch_std": raw_std,
            "canonical_initial_batch_std": canonical_std,
            "initial_batch_std_reduction": std_reduction,
        }
    )

rows


## Inspect one backend-native equation summary

These are backend-native PySINDy terms, not canonical PDE terms.
If the sparse equations are empty under the frozen runtime threshold, inspect the top dense coefficient magnitudes instead.


In [ ]:
print("inspected feature:", primary_feature)
print("raw initial-time batch std:", raw_primary_std)
print("canonical initial-time batch std:", canonical_primary_std)
print("initial-time batch std reduction:", primary_std_reduction)
print()
print("raw equation string:")
print(raw["fit"]["equation_strings"][primary_feature])
print()
print("canonical equation string:")
print(canonical["fit"]["equation_strings"][primary_feature])
print()
print("top raw dense coefficients:")
print(top_dense_terms(raw["fit"], primary_feature))
print()
print("top canonical dense coefficients:")
print(top_dense_terms(canonical["fit"], primary_feature))
